In [0]:
from config.f1_config import add_ingestion_metadata,CATALOG, BRONZE_SCHEMA, SILVER_SCHEMA
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType,TimestampType, FloatType
import pyspark.sql.functions as F

SILVER_TABLE = "f1_workspace.silver.circuits"

In [0]:
df_bronze_circuits = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.circuits")
display(df_bronze_circuits)

TRASFORM AND FLATTEN DATA


In [0]:
def transform_circuits_silver(df_bronze):
    
    return (
        df_bronze.select(
            F.trim(F.col("circuitId")).alias("Circuit_Id"),
            F.trim(F.initcap(F.col("circuitName"))).alias("Circuit_Name"),
            F.trim(F.col("Location.long")).cast("double").alias("Longitude"),
            F.trim(F.col("Location.lat")).cast("double").alias("Latitude"),
            F.trim(F.initcap(F.col("Location.locality"))).alias("City"),
            F.trim(F.initcap(F.col("Location.country"))).alias("Country"),
        )
        .withColumn(
            "Country",
             F.when(F.col("Country") == "Usa", "USA")
             .when(F.col("Country") == "Uk", "UK")
             .when(F.col("Country") == "Uae", "UAE")
             .otherwise(F.col("Country"))
        )
        .withColumn("silver_processed_timestamp", F.current_timestamp())
    )


df_silver_circuits = transform_circuits_silver(df_bronze_circuits)


In [0]:
df_silver_circuits.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(SILVER_TABLE)